# Running [passim](https://github.com/dasmiq/passim) on a small corpus (OpenITI configuration)

This notebook runs passim on a set of texts. It is intended to be a no-code interface for running passim (you just need to upload files into colab and press the play buttons on each code block). You are, however, encouraged to make a copy of this notebook and adapt it for your needs (for this more advanced option some knowledge of Python and the Command Line Interface is required)

It first takes the text files to insert milestone tags inside the text to use them for chunking the texts. The texts are then fed into passim in those chunks. If you wish to chunk your text differently, or to run passim on a full-length text, you will need to change the code accordingly.

To start the script, upload your texts into this notebook, either in the root folder or in a new folder. Then when the script asks for the input texts, you give the path to the texts. Make sure you name the text files with proper, unique book ids so that you can identify those books ids in the passim text-reuse data. We suggest to keep the same structure of filename.txt for all text files as the script takes whats comes before ".txt" as the book ids and uses them later in out folder and file names. For file names, use a combinaton of letters and numbers and avoid special characters, whitespaces, and any unconventional letters as much as possible.

The first code block installs the openiti library.

In [ ]:
!pip install openiti

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.7 MB/s eta 0:00:00
  Created wheel for pypyodbc: filename=pypyodbc-1.3.6-py3-none-any.whl size=22857 sha256=96483e19df4759cdf1f2641ef133c4e533ab3c7dc62a306b165f7436c66d3a37
  Stored in directory: /root/.cache/pip/wheels/cc/0f/d7/04afb4d86f85c4969168e5a366dbf8c17751159188c2c677ed
Successfully built pypyodbc


# Adding milestone tags to the input texts

The script in the code block below inserts milestone tags into the original text files. If you want to keep your original files and write the texts with milestone tags in new files, make the required changes by giving a new path to the script to write in the code below.

You can specify the length of the chunks (milestones) in words as as input or hard code it (KITAB chunks OpenITI texts into 300 word pieces for its passim runs).


An example from OpenITI of a milestone marker (ms173) in a text:
<blockquote>
لو انه أغنى بكيت كخندف % على الياس حتى ملها السر تندب
 إذا مؤنس لاحت خراطيم شمسه % بكت غدوة حتى ترى الشمس تغرب
 يعني بقوله مؤنس يوم الخميس لأن العرب كانت تسمي الأيام بغير أسمائها في
هذا الوقت فكانت تسمي الأحد الأول والاثنين أهون والثلاثاء جبار والاربعاء
دبار والخميس مؤنسا والجمعة عروبة والسبت شيار وكانوا يسمون أيام الشهر <mark style ="background-color:#0000FF">ms173</mark>
عشرة أسماء كل ثلاث ليال اسم فالثلاث التي أول الهلال الغرر ثم النفل ثم
التسع ثم العشر ثم البيض ثم الظلم ثم الخنس ثم الحنادس ثم المحاق والآخر
ليلة السرار إذا استسر الهلال وكانوا يسمون المحرم مؤتمرا وصفرا ناجرا
وربيعا الأول خوان وربيعا الآخر وبصان وجمادى الأول حنين وجمادى الآخرة ربى
ورجبا الأصم وشعبان العاذل ورمضان ناتقا وشوالا وعلا وذا القعدة ورنة وذا
الحجة بركا وكان آخرون من العرب يسمون الثلاث ليال من أول الشهر هلالا ثم
ثلاث قمر حين يقمر ثم ثلاث بهر حين يضيء ويبهر لونه وثلاث نقل وثلاث بيض
وثلاث درع وثلاث ظلم وثلاث حنادس وثلاث دآدي وليلتان محاق وليلة سرار
وولد لطانجة بن إلياس اد فتفرقت من ولد اد بن طانجة أربع
</blockquote>

The script also uses the header-body splitter pattern to perform the process on only the body text in case texts have any splitter. The default pattern is the one that is being used in OpenITI corpus. For other existing patterns in the input texts, change the splitter value in the scripts below. The script also works if the text does not have any splitter and takes the whole content of the text file as the body text.

The default pattern for milestone tags are "\s{1}ms[A-Z]?\d+". For a different pattern, change the constant value or make an input variable in the code.

In [ ]:
# Masoumeh's passim input script

#!/usr/bin/env python3
"""
Insert fixed-length milestones into text files based on Arabic token counts.
"""

import os
import re
import sys
import math
from collections import Counter
from itertools import groupby
from typing import Optional
import openiti.helper.ara as ara

# Constants
SPLITTER = "#META#Header#End#"
FILE_PATTERN = re.compile(r".*(\.txt)$")
MS_FIND_REGEX = re.compile(r"ms[A-Z]?\d+")
MS_REMOVE_REGEX = re.compile(r"\s{1}ms[A-Z]?\d+")
MILESTONE300_REGEX = re.compile(r"\s*Milestone300")
# Set to True to process files even when the SPLITTER is missing
PROCESS_WITHOUT_SPLITTER = True


def insert_milestones(
    filepath: str, length: int, last_ms: int, log_file
) -> Optional[int]:
    """
    Read `filepath`, strip old milestone tags, and insert new ones every `length`
    Arabic tokens, continuing count from `last_ms`. Returns final ms count,
    or None on error.
    """
    # filename = os.path.basename(filepath)
    # name_base = re.split(r"-[a-z]{3}\d", filename)[0]

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Split header/body, optionally skipping the missing‐splitter error
    if SPLITTER in content:
        header, body = map(str.rstrip, content.split(SPLITTER, 1))
        log_file.write(f"[DONE] Splitter in {filepath}, milestoned!\n")
    elif not PROCESS_WITHOUT_SPLITTER:
        log_file.write(f"[ERROR] Missing splitter in {filepath}, no changes made!\n")
        return None
    else:
        # No splitter but we're configured to proceed:
        header = ""
        body = content#.rstrip()
        log_file.write(f"[WARN] Missing splitter in {filepath}, milestoned!\n")

    # Remove old milestone tags, including the old pattern (Milestone300)
    body = MS_REMOVE_REGEX.sub("", body)
    body = MILESTONE300_REGEX.sub("", body)

    # Check for stray IDs
    # remaining = MS_FIND_REGEX.findall(body)
    # if len([m for m in remaining if m != ""]) > 1:
    #     log_file.write(f"[ERROR] Remaining IDs in {filepath}: {remaining}\n")
    #     return None

    # Count Arabic tokens
    total_ar = ara.ar_tok_cnt(body)
    pad_len = len(str(math.floor(total_ar / length)))
    tokens = re.findall(r"\w+|\W+", body)

    count, ms_count = 0, last_ms
    new_parts = []

    for i, tok in enumerate(tokens):
        new_parts.append(tok)
        if re.search(ara.ar_tok, tok):
            count += 1

        # Time to insert a milestone?
        if count >= length or i == len(tokens) - 1:
            ms_count += 1
            tag = f" ms{ms_count:0{pad_len}d}"
            new_parts.append(tag)
            count = 0

    new_body = "".join(new_parts)

    # Verify no corruption
    if MS_REMOVE_REGEX.sub("", new_body) != body:
        log_file.write(f"[ERROR] Content mismatch in {filepath}\n")
        return None

    # Check for duplicates
    ids = MS_FIND_REGEX.findall(new_body)
    dupes = {k: v for k, v in Counter(ids).items() if v > 1}
    if dupes:
        log_file.write(f"[ERROR] Duplicate IDs in {filepath}: {dupes}\n")
        return None

    # Write out
    if SPLITTER in content:
      final = header + "\n\n" + SPLITTER + "\n\n" + new_body
    else:
      final = new_body
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(final)

    return ms_count


def process_files(root_folder: str, ms_length: int) -> None:
    """
    Walk `root_folder`, find matching text files, and insert milestones.
    """
    if not os.path.isdir(root_folder):
        print(f"Invalid path: {root_folder}")
        sys.exit(1)

    log_path = os.path.join(root_folder, "../milestone_log.txt")
    with open(log_path, "w", encoding="utf-8") as log_file:
        for root, _, files in os.walk(root_folder):
            # Filter book files
            book_files = [
                f for f in files if FILE_PATTERN.match(f)
            ]
            for fname in book_files:
                insert_milestones(os.path.join(root, fname), ms_length, 0, log_file)

# def main():
folder = input("Enter the path to the OpenITI folder: ").strip()
try:
    length = int(input("Enter the length of milestones: ").strip())
except ValueError:
    print("Milestone length must be an integer.")
    sys.exit(1)

process_files(folder, length)
print("Done!")

Enter the path to the OpenITI folder: corpus
Enter the length of milestones: 300
Done!


# Transforming the texts into an input file ([JSON Lines format](https://jsonlines.org//))

This script in the code block below generates input files for passim using the inputs as prepared, either chunks of text or full-length texts. It is recommended to use chunked texts (especially if you expect significant rearrangment of aligned text between the books being compared).

The generated data is the default input format for documents and is in a file or set of files containing one JSON record per line, i.e., the JSON Lines format. For large size of corpus, to avoid big input files, we set a threshold (in the below snippet it is 1000) to limit the number of JSON records per file.

The record for a single document with the required `id` and `text` fields, as well as a `series` field, would look like:
```
{"id": "d1", "series": "abc", "text": "This is text."}
```

In addition to the above fields, other metadata included in the record for each document will be passed through into the output.


In [ ]:
# prepare passim inputs
import os
import re
import sys
import csv
from itertools import groupby

import pandas as pd
from openiti.helper import funcs, ara


def mechanical_chunking(filename, text, output_dir, milestone_pattern, chunk_size):
    """
    Split `text` on each occurrence of `milestone_pattern`, clean it,
    and write JSON-like chunks of size `chunk_size` to files in `output_dir`.
    Returns 1 if any chunks were written, 0 if the last expected chunk already exists.
    """
    # Derive a simple book ID and prepare output path prefix
    book_id = re.sub(r"(\.txt)$", "", filename)
    print(book_id)
    prefix = os.path.join(output_dir, book_id)

    # If the final chunk file already exists, skip
    final_path = f"{prefix}-{chunk_size:05d}"
    if os.path.exists(final_path):
        return 0

    # Split on the milestone markers, keeping them in the list
    parts = re.split(fr"({milestone_pattern})", text)
    template = '{{"id":"{id}", "series":"{series}", "text":"{text}", "seq":{seq}}}'
    records = []
    counter = 0
    i = 0

    while i < len(parts) - 2:
        counter += 1
        marker = parts[i + 1]
        # remove vowels
        clean_text = ara.denoise(parts[i])
        # clean text
        clean_text = funcs.text_cleaner(clean_text)

        seq = int(re.sub(r"\D", "", marker))
        rec = template.format(
            id=f"{book_id}.{marker}",
            series=book_id,
            text=clean_text,
            seq=seq
        )
        records.append(rec)

        # Write out in batches of chunk_size
        if counter % chunk_size == 0:
            path = f"{prefix}-{counter:05d}.json"
            with open(path, "w", encoding="utf8") as fw:
                fw.write("\n".join(records))
            records = []

        i += 2

    # Write any remaining records
    # Round up to the next multiple of chunk_size
    final_counter = ((counter + chunk_size - 1) // chunk_size) * chunk_size
    path = f"{prefix}-{final_counter:05d}.json"
    with open(path, "w", encoding="utf8") as fw:
        fw.write("\n".join(records))

    return 1


main_folder = input("Enter the path to the text folder: ").strip()
target_folder = "passim_inputs" #input("Enter the path to write the new files: ").strip()
os.makedirs(target_folder, exist_ok=True)

# Validate paths
if not os.path.exists(main_folder):
  print(main_folder)
  print(f"Invalid path: {main_folder}", file=sys.stderr)
  sys.exit(1)

print("\nGenerating mechanical passim corpus...\n")

# Constants
milestone = r"ms[A-Z]?\d+"
SPLITTER = "#META#Header#End#"
chunk_size = 1000
written = []
total = 0

for root, _, files in os.walk(main_folder):
    # Match files like filename.txt
    print(files)
    pattern = r".+(\.txt)$"
    candidates = [f for f in files if re.match(pattern, f)]
    if not candidates:
        continue

    # Process each file
    for filename in candidates:
        path = os.path.join(root, filename)
        with open(path, encoding="utf8") as file:
            content = file.read()
            if SPLITTER in content:
              body = content.split(SPLITTER)[1]
            else:
              body = content

        status = mechanical_chunking(
            filename, body, target_folder, milestone, chunk_size
        )
        total += status
        book_id = re.sub(r"(\.txt)$", "", filename)
        written.append((book_id, status))

        if total % 100 == 0:
            print(f"\nProcessed: {total}\n" + "=" * 20 + "\n")

# Write a log of what was written
with open("corpus_log.csv", "w", newline="", encoding="utf8") as log_f:
    writer = csv.writer(log_f)
    writer.writerow(["book_id", "write_status"])
    writer.writerows(written)

print("Done!")


Enter the path to the text folder: corpus

Generating mechanical passim corpus...

['0845Maqrizi.Muqaffa.Sham19Y0145334-ara1.completed.txt', '0660IbnCadim.BughyatTalab.Shamela0010798-ara1.mARkdown.txt', '0542IbnMunjibTajRiyasaIbnSayrafi.Ishara.MAB09032026-ara1.txt', '0845Maqrizi.Mawaciz.Shamela0011566-ara1.mARkdown.txt']
0845Maqrizi.Muqaffa.Sham19Y0145334-ara1.completed
0660IbnCadim.BughyatTalab.Shamela0010798-ara1.mARkdown
0542IbnMunjibTajRiyasaIbnSayrafi.Ishara.MAB09032026-ara1
0845Maqrizi.Mawaciz.Shamela0011566-ara1.mARkdown
Done!


# Install passim

The code block below downloads passim using python's pip installer.

In [ ]:
# Install passim

!pip install git+https://github.com/dasmiq/passim.git

  Cloning https://github.com/dasmiq/passim.git to /tmp/pip-req-build-0z919_x1
  Running command git clone --filter=blob:none --quiet https://github.com/dasmiq/passim.git /tmp/pip-req-build-0z919_x1
  Resolved https://github.com/dasmiq/passim.git to commit e4eea1c89c4182094d119047c85032cd9163ea17
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.4 MB/s eta 0:00:00
  Created wheel for passim: filename=passim-2.0.0-py3-none-any.whl size=15556 sha256=cf3e1b8078b5a072c2a43d4b20ebec1f454172cca6716975bf646c2e2fe1124e
  Stored in directory: /tmp/pip-ephem-wheel-cache-7yg6whfm/wheels/b0/51/a2/50fb4f12befcc05270d5ad947734813b1e14df43e9f21862ad
Successfully built passim


# Run passim

The below code block runs passim invoking it through SPARK (the same can be done locally in the command line with passim installed).

The code first removes the existing path to the output folder ("passim_output_json" in the below example) to overwrite the old output in each run. By default it takes the input folder produced by the script above. You can change the output folder as it fits your folder structure, but the current argument gives an output that can be processed by the post-processing script without any changes.

Current arguments:

```
--pairwise --filterpairs 'gid < gid2'
```
In the below run, `--pairwise` argument invokes passim to output pairwise alignments between all matching passages (in addition to the cluster data).

For full documentation of passim and how to run, see https://github.com/dasmiq/passim.

This script is designed to run cleanly, cleaning up any previous passim outputs in your file system before running passim. If you are running this notebook for the first time, or have relaunched the runtime, then this script will likely return the below message:

```
rm: cannot remove 'passim_output_json/': No such file or directory
```
This is a warning and can be ignored - it will not stop passim from running.

In [ ]:
# FROM DAVID'S NOTEBOOK: https://colab.research.google.com/github/dasmiq/passim/blob/main/docs/passim_quickstart.ipynb#scrollTo=hZBefy-B1S01
# This takes input files in passim_inputs dir (created by the above steps)


# Delete old output
!rm -r passim_output_json/
!SPARK_SUBMIT_ARGS="--driver-memory 8G --executor-memory 8G" passim --pairwise --filterpairs 'gid < gid2' passim_inputs passim_output_json >& out_cluster.err

rm: cannot remove 'passim_output_json/': No such file or directory


# Convert the output files to pairwise csvs compatible with KITAB apps

The code block below takes the output from passim (which is in json format) and converts it to a [tab separated values file (tsv) ](https://en.wikipedia.org/wiki/Tab-separated_values) that can be loaded into KITAB apps. However, **the output files have ".csv" extension. To open them in any spreadsheet software, make sure to use `Tab` as the separator, not comma!**

The columns include:

* begin: begin point of alignment in characters
* begin2: begin point of alignment2 in characters
* end: end point of alignment in characters
* end2: end point of alignment2 in characters
* gid: hash of series field
* gid2: hash of series2 field
* id: full id of text chunk, including book id and milestone id in book 1
* id2: full id of text chunk 2, including book id and milestone id in book 2
* matches: number of matches in aligned sequences
* s1: aligned sequence in book 1
* s2: aligned sequence in book 2
* seq: chunk (milestone) id in book 1. This value has been included as metadata in the record for each document in passim input in the preparation process and then is passed through into the output.
* seq2: chunk (milestone) id in book 2. This value has been included as metadata in the record for each document in passim input in the preparation process and then is passed through into the output.
* series: series identifier
* series2: series identifier for text 2
* uid: hash of document id in book 1
* uid2: hash of document id in book 2

Following columns are added by the post-procesing script:

* w_match: count of words matched in the alignment (excluding whitespace)
* ch_match: count of characters matched in the alignment
* align_len: length of the alignment in s1
* matches_percentage: percentage of words match in s1


When the script has run, you will find the outputs in the 'pairwise-alignments' folder. You can download this file and read it directly in excel or google sheets. Alternatively, you can upload it to the [KITAB diffviewer](https://kitab-project.org/diffViewer/) to study the alignments with differences highlighted (choose the 'upload from file' option and upload your tsv).

As the output is a tsv and contains Arabic script, to open it in excel you will need to import it through the data tab. To do so follow these steps:
1. Open excel and create a new spreadsheet
1. Go to the 'Data' tab
1. Click 'Get Data' --> 'From File' --> 'From Text/CSV'
1. Select the passim tsv file that you have downloaded from colab and click 'import'
1. A modal will appear (65001:Unicode (UTF-8) should appear in the top left dropdown) - click 'Load'
1. Your data will load into excel

For larger outputs (greater than 300 rows) we recommend uploading the tsv file to google drive and viewing it using google sheets (this will be more stable).


In [ ]:
# Masoumeh's script for outputs

# It will take ''out_cluster/out.json' as input

#!/usr/bin/env python3
import pandas as pd

"""
This script augments Passim outputs (JSON or Parquet) with:
  - ch_match: character matches excluding whitespace
  - align_len: length of the aligned string
  - matches_percentage: (matches / align_len) * 100
  - w_match: word matches at identical positions

Outputs are written as partitioned CSVs by series/series2,
and post-processed to remove hidden files and rename parts.
"""

import os
import re
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType
import pyspark.sql.functions as F

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURE YOUR INPUT & OUTPUT PATHS
# ──────────────────────────────────────────────────────────────────────────────
INPUT_PATH = "passim_output_json/align.json"
OUTPUT_PATH = "passim_output_csv/pairwise-alignments"  # directory where CSVs will be written
# ──────────────────────────────────────────────────────────────────────────────


def word_count(s1: str, s2: str) -> int:
    """Count matching words between two aligned strings at identical positions."""
    cnt = 0
    i, n = 0, len(s1)
    while i < n:
        start = i
        while i < n and s1[i] != " ":
            i += 1
        if s1[start:i] == s2[start:i]:
            cnt += 1
        i += 1  # skip the space
    return cnt


def ch_count(s1: str, s2: str) -> int:
    """Count matching non-whitespace characters at identical positions."""
    return sum(1 for a, b in zip(s1, s2) if a == b and a != " ")


def match_percentage(matches: int, length: int) -> float:
    """Compute match percentage; return 0.0 if length is zero."""
    return (matches / length) * 100 if length > 0 else 0.0


def process_alignment(input_path: str, output_path: str) -> None:
    spark = SparkSession.builder.appName("Stats").getOrCreate()

    # Register UDFs
    word_match_udf = F.udf(word_count, IntegerType())
    ch_match_udf = F.udf(ch_count, IntegerType())
    align_len_udf = F.udf(lambda s: len(s), IntegerType())
    percent_udf = F.udf(match_percentage, FloatType())

    # Determine format from extension
    fmt = Path(input_path).suffix.lower()
    if fmt == ".json":
        src_format = "json"
    elif fmt == ".parquet":
        src_format = "parquet"
    else:
        raise ValueError(f"Unsupported input format: {fmt}")

    # Load, dedupe, and drop nulls
    df = (
        spark.read
             .format(src_format)
             .option("encoding", "UTF-8")
             .load(input_path)
             .distinct()
             .na.drop()
    )
    # new_col_names = {'id':'id1', 'first': 'first1', 'uid':'uid1', 'bw':'bw1',
    #                 'ew': 'ew1', 'begin':'b1', 'begin2':'b2', 'end':'e1',
    #                 'end2': 'e2', 'len': 'len1',
    #                 'tok': 'tok1', 'seq': 'seq1', 'gid': 'gid1'
    #                 , 'series': 'series1', 'mathces_percentage': 'matches_percent'}
    #                 #  , 'series2': 'series_b2'}

    # for old_name, new_name in new_col_names.items():
    #     df = df.withColumnRenamed(old_name, new_name)

    # Compute new columns
    df2 = (
        df
        .withColumn("ch_match", ch_match_udf("s1", "s2"))
        .withColumn("align_len", align_len_udf("s1"))
        .withColumn(
            "matches_percentage",
            F.when(F.col("align_len") == 0, F.lit(0.0))
             .otherwise(percent_udf(F.col("matches"), F.col("align_len")))
        )
        .withColumn("w_match", word_match_udf("s1", "s2"))
        .withColumn('series_b1', F.col('series')) \
        .withColumn('series_b2', F.col('series2')) \

    )

    # Write partitioned CSV output
    df2.repartition("series", "series2") \
       .sortWithinPartitions("id", "id2") \
       .write \
       .partitionBy("series", "series2") \
       .format("csv") \
       .option("header", "true") \
       .option("delimiter", "\t") \
       .mode("overwrite") \
       .save(output_path)

    spark.stop()



    # Post-process: remove hidden files, rename part files, and clean directories
    for root, _, files in os.walk(output_path, topdown=False):
        root_path = Path(root)

        # Inside a series2 partition
        if any(p.startswith("series2=") for p in root_path.parts):
            # Delete hidden files (e.g. ._SUCCESS)
            for f in files:
                if f.startswith("."):
                    (root_path / f).unlink()

            # Rename the first part file to SERIES_SERIES2.csv
            parts = [f for f in files if f.startswith("part")]
            if parts:
                series = next((p.split("=", 1)[1] for p in root_path.parts if p.startswith("series=")), "")
                series2 = next((p.split("=", 1)[1] for p in root_path.parts if p.startswith("series2=")), "")
                new_name = f"{series}_{series2}.csv"
                parent = root_path.parent
                os.rename(root_path / parts[0], parent / new_name)

            # Remove the now-empty series2 directory
            root_path.rmdir()

        # Rename the series directory to drop the prefix
        elif any(p.startswith("series=") for p in root_path.parts):
            new_dir = re.sub(r"series=", "", str(root_path))
            os.rename(root, new_dir)


# if __name__ == "__main__":
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(INPUT_PATH)
if not os.path.exists(INPUT_PATH):
  print(f"Invalid path: {INPUT_PATH}", file=sys.stderr)
  sys.exit(1)
process_alignment(INPUT_PATH, OUTPUT_PATH)



passim_output_json/align.json


# Credits

The idea for this notebook was adapted from a resource created by David Smith (the creator of passim). The original notebook can be found [here](https://github.com/dasmiq/passim/blob/main/docs/passim_quickstart.ipynb)

Scripts for creating passim inputs and processing outputs according to OpenITI schemas were developed by Masoumeh Seydi as part of the KITAB project (they are based on the processing scripts used by the team in their routine passim runs). If these scripts are reused outside of this notebook, it is recommended that you cite Masoumeh (for example, in the code comments or in the doc string of the relevant functions).

The documentation in this notebook was composed jointly by Masoumeh Seydi and Mathew Barber.

Thank you in advance for crediting the hard work of the team!